In [ ]:
from sklearn.metrics import mean_squared_error
import pandas as pd
import warnings
from sklearn import preprocessing
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

# from lightgbm import LGBMRegressor
from sklearn import datasets
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import time
import torch
import torch
import numpy as np
import numpy as np

np.random.seed(42)
torch.manual_seed(42)
warnings.filterwarnings("ignore")
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import matplotlib
import pickle
matplotlib.rcParams['font.sans-serif'] = ['SimHei']  
matplotlib.rcParams['axes.unicode_minus'] = False 
import geopandas as gpd
import xgboost as xgb
from sklearn.metrics import mean_squared_error
import numpy as np
from osgeo import gdal
import numpy as np
import os
import xgboost as xgb
import numpy as np
from sklearn.metrics import accuracy_score
import time
import numpy as np
from collections import Counter
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, mean_squared_error
import joblib
from flaml import AutoML
from sklearn.metrics import accuracy_score
import os
import pickle


d:\Anaconda3\envs\ddl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
"""
The data volume is too large to upload. Please refer to the run history.
"""

## Get training data

In [ ]:



def train_AutoML_Classification(train_data,train_labels, modelPath=None, IsValidation=False,Minute=60):
    
    settings = {
        "time_budget": Minute * 60,         
        "metric": "accuracy",           
        "estimator_list": 'auto',        
        "task": "classification",       
        "eval_method": "cv",            
        "n_splits": 5,                   
        "early_stop": True,              
        "verbose": 0,
        "seed": 42
    }
    if not IsValidation:
        automl = AutoML()
        automl.fit( X_train=train_data,  y_train=train_labels,  **settings  )
        
        if modelPath:
            with open(modelPath, "wb") as f:
                pickle.dump(automl, f)
    else:
        with open(modelPath, "rb") as f:
            automl = pickle.load(f)

    # 预测与评估
    y_pred = automl.predict(train_data)
    acc = accuracy_score(train_labels, y_pred)

    print(f"Training Accuracy: {acc:.4f}")

    return automl



def train_AutoML_Regression(train_data, train_labels, modelPath=None, IsValidation=False, Minute=60):
    
    # Settings for AutoML regression task
    settings = {
        "time_budget": Minute * 60,          # Total training time (in seconds)
        "metric": "mse",                   # Use RMSE as the metric for regression
        "estimator_list": 'auto',           # "auto" allows the AutoML to choose the best estimator
        "task": "regression",               # Set task to regression
        "eval_method": "cv",                # Enable cross-validation
        "n_splits": 5,                      # 5-fold cross-validation
        "early_stop": True,                 # Enable early stopping
        "verbose": 0,
        "seed": 42
    }
    
    if not IsValidation:
        # Initialize and fit AutoML for regression task
        automl = AutoML()
        automl.fit(X_train=train_data, y_train=train_labels, **settings)

        # Save the trained model if `modelPath` is provided
        if modelPath:
            with open(modelPath, "wb") as f:
                pickle.dump(automl, f)
    else:
        # Load the pre-trained model from file if `IsValidation` is True
        with open(modelPath, "rb") as f:
            automl = pickle.load(f)

    # Predictions and evaluation
    y_pred = automl.predict(train_data)
    
    # RMSE for training predictions
    rmse = np.sqrt(mean_squared_error(train_labels, y_pred))
    print(f"Training RMSE: {rmse:.4f}")
    return automl






In [ ]:
usingFeatures=["DEM_CTSE", "DEM_GDEM",  "FC", "FH", "FH10", "FH30","BH90", "FCLoss",  "DEM", "Cur", "Aspect", "Hills", "HEL", "NIR",  "RGB","SAR","deltMin"]


def genTrainingData_RegressionBias(shp_path,GDEM):
    
    gdf = gpd.read_file(shp_path)
    
 
    columns_to_check = ["CTSE", GDEM]  
    df = gdf[(gdf[columns_to_check] > 0).all(axis=1)]

    df['DEM_EGM2008']=df['DEM']-df['EGM2008']
    mean_value = df['DEM_EGM2008'].mean()
    std_value = df['DEM_EGM2008'].std()

    lower_bound = mean_value - 3 * std_value
    upper_bound = mean_value + 3 * std_value

    df = df[(df['DEM_EGM2008'] >= lower_bound) & (df['DEM_EGM2008'] <= upper_bound)]

 

    CTSEArray=np.array(df['CTSE'])
    GDEMModelArray=np.array(df[GDEM])

    ICEeleArray = np.array(df['EGM2008'])
    DEMArray = np.array(df['DEM'])

    df['DEM_CTSE']=df['DEM']-df['CTSE']
    df['DEM_GDEM']=df['DEM']-df[GDEM]
   
  
    df['label'] = df['DEM']-df['EGM2008']
    labels = np.array(df['label'])

    df=df[usingFeatures]

    input_features = np.array(df)
    scaler = StandardScaler()
    input_features = scaler.fit_transform(input_features)  

    train_data = np.array(input_features)
    train_labels = np.array(labels)

    test_CTSE = np.array(CTSEArray)
    test_GDEM= np.array(GDEMModelArray)

    test_ICE = np.array(ICEeleArray)
    test_DEM = np.array(DEMArray)
    

    GDEMDict={}
    GDEMDict['CTSE']=test_CTSE
    GDEMDict['GDEM']=test_GDEM
    
    GDEMDict['ICE']=test_ICE
    GDEMDict['DEM']=test_DEM

    return train_data,train_labels,GDEMDict,scaler




def genTrainingData_Classification(shp_path,GDEM):
    
    gdf = gpd.read_file(shp_path)
    
    columns_to_check = ["CTSE", GDEM]  
    df = gdf[(gdf[columns_to_check] > 0).all(axis=1)]

    df['DEM_EGM2008']=df['DEM']-df['EGM2008']
    mean_value = df['DEM_EGM2008'].mean()
    std_value = df['DEM_EGM2008'].std()

    lower_bound = mean_value - 3 * std_value
    upper_bound = mean_value + 3 * std_value

    df = df[(df['DEM_EGM2008'] >= lower_bound) & (df['DEM_EGM2008'] <= upper_bound)]

    df['NSCT_GDEM']=(df['CTSE']-df[GDEM]).abs()
    

    diffs = pd.DataFrame({
        0: (df['EGM2008'] - df['CTSE']).abs(),
        1: (df['EGM2008'] - df[GDEM]).abs()
    })

    
    CTSEArray=np.array(df['CTSE'])
    GDEMModelArray=np.array(df[GDEM])

    ICEeleArray = np.array(df['EGM2008'])
    DEMArray = np.array(df['DEM'])

    df['DEM_CTSE']=df['DEM']-df['CTSE']
    df['DEM_GDEM']=df['DEM']-df[GDEM]
   
 
    df['label'] = diffs.idxmin(axis=1)
    labels = np.array(df['label'])

    df=df[usingFeatures]

    input_features = np.array(df)


    input_features = np.array(df)  
    scaler = StandardScaler()
    input_features = scaler.fit_transform(input_features) 

    train_data = np.array(input_features)
    train_labels = np.array(labels)

    test_CTSE = np.array(CTSEArray)
    test_GDEM= np.array(GDEMModelArray)

    test_ICE = np.array(ICEeleArray)
    test_DEM = np.array(DEMArray)
    
    GDEMDict={}
    GDEMDict['CTSE']=test_CTSE
    GDEMDict['GDEM']=test_GDEM
    
    GDEMDict['ICE']=test_ICE
    GDEMDict['DEM']=test_DEM

    return train_data,train_labels,GDEMDict, scaler



def genTrainingData_RegressionCoeff(shp_path,GDEM):
    
    gdf = gpd.read_file(shp_path)
    
    columns_to_check = ["CTSE", GDEM]  
    df = gdf[(gdf[columns_to_check] > 0).all(axis=1)]

    df['DEM_EGM2008']=df['DEM']-df['EGM2008']
    mean_value = df['DEM_EGM2008'].mean()
    std_value = df['DEM_EGM2008'].std()

    lower_bound = mean_value - 3 * std_value
    upper_bound = mean_value + 3 * std_value

    df = df[(df['DEM_EGM2008'] >= lower_bound) & (df['DEM_EGM2008'] <= upper_bound)]

  
    df['EGM2008_to_CTSE_squared'] = (df['EGM2008'] - df['CTSE']) ** 2
    df['EGM2008_to_GDEM_squared'] = (df['EGM2008'] - df[GDEM]) ** 2

    df['CTSE_coeff'] = df['EGM2008_to_GDEM_squared'] / (df['EGM2008_to_CTSE_squared'] + df['EGM2008_to_GDEM_squared']) # GDEM as fenzi

    labels = np.array(df['CTSE_coeff'])

    CTSEArray=np.array(df['CTSE'])
    GDEMModelArray=np.array(df[GDEM])

    ICEeleArray = np.array(df['EGM2008'])
    DEMArray = np.array(df['DEM'])

    df['DEM_CTSE']=df['DEM']-df['CTSE']
    df['DEM_GDEM']=df['DEM']-df[GDEM]

    df=df[usingFeatures]

    input_features = np.array(df)
    scaler = StandardScaler()
    input_features = scaler.fit_transform(input_features)  


    train_data = np.array(input_features)
    train_labels = np.array(labels)

    test_CTSE = np.array(CTSEArray)
    test_GDEM= np.array(GDEMModelArray)

    test_ICE = np.array(ICEeleArray)
    test_DEM = np.array(DEMArray)
    
    GDEMDict={}
    GDEMDict['CTSE']=test_CTSE
    GDEMDict['GDEM']=test_GDEM
    
    GDEMDict['ICE']=test_ICE
    GDEMDict['DEM']=test_DEM

    return train_data,train_labels,GDEMDict, scaler




##  Raster Prediction

In [ ]:

def AssignValuesByPixelID(input_raster_path, output_raster_path, pid_array, value_array):
    dataset = gdal.Open(input_raster_path, gdal.GA_ReadOnly)
    band = dataset.GetRasterBand(1)
    pixel_array = band.ReadAsArray()
    nodata = band.GetNoDataValue()



    if nodata is not None:
        pixel_array = np.where(pixel_array == nodata, 0, pixel_array)  

    pixel_id_array = pixel_array.astype(np.int32)


    lut_size = pixel_id_array.max() + 1
    
    lut = np.full(lut_size, nodata, dtype=np.float32)


    pid_array = pid_array.astype(np.int32)

    lut[pid_array] = value_array


    mapped_array = lut[pixel_id_array]

    driver = gdal.GetDriverByName('GTiff')
    outDataset = driver.Create(output_raster_path,
                               pixel_id_array.shape[1], pixel_id_array.shape[0],
                               1, gdal.GDT_Float32)
    outDataset.SetGeoTransform(dataset.GetGeoTransform())
    outDataset.SetProjection(dataset.GetProjection())
    outBand = outDataset.GetRasterBand(1)
    outBand.WriteArray(mapped_array)
    outBand.SetNoDataValue(nodata)
    outBand.FlushCache()

    dataset, band, outDataset, outBand = None, None, None, None
    return output_raster_path


def finalPrediction(df_Path,model,PidRaster,out_fusion,GDEM):

    df_pred = pd.read_json(df_Path)
    columns_to_check = ["CTSE",GDEM ] 
    df_pred = df_pred[(df_pred[columns_to_check] > 0).all(axis=1)]
    
    PidArray = np.array(df_pred['Pid'])
    CTSEArray = np.array(df_pred['CTSE'])
    GDEMArray = np.array(df_pred[GDEM])
    
    df_pred['DEM_CTSE']=df_pred['DEM']-df_pred['CTSE']
    df_pred['DEM_GDEM']=df_pred['DEM']-df_pred[GDEM]
   
    df_pred=df_pred[usingFeatures]

    input_features = np.array(df_pred)
    input_features = preprocessing.StandardScaler().fit_transform(input_features)

    Predslabels = model.predict(input_features)


    stacked_sources = np.vstack([CTSEArray, GDEMArray]) 

    PredsArray = stacked_sources[Predslabels.astype(int), np.arange(len(Predslabels))]  

    AssignValuesByPixelID(input_raster_path=PidRaster, output_raster_path=out_fusion, pid_array=PidArray, value_array=PredsArray)


def finalPrediction_Prob(scaler, df_Path,model,PidRaster,out_fusion,GDEM):

    df_pred = pd.read_json(df_Path)
    columns_to_check = ["CTSE",GDEM ]  
    df_pred = df_pred[(df_pred[columns_to_check] > 0).all(axis=1)]
    
    PidArray = np.array(df_pred['Pid'])
    CTSEArray = np.array(df_pred['CTSE'])
    GDEMArray = np.array(df_pred[GDEM])
    DEMArray = np.array(df_pred['DEM'])
    
    df_pred['DEM_CTSE']=df_pred['DEM']-df_pred['CTSE']
    df_pred['DEM_GDEM']=df_pred['DEM']-df_pred[GDEM]

    PredError_CTSE=np.array(df_pred['DEM_CTSE'])
    PredError_GDEM=np.array(df_pred['DEM_GDEM'])
   
    df_pred=df_pred[usingFeatures]

    input_features = np.array(df_pred)
    input_features = scaler.transform(input_features)  
    
    probs = model.predict_proba(input_features)
    predErrors_fused = PredError_CTSE * probs[:, 0] + PredError_GDEM * probs[:, 1]

    PredsArray=DEMArray-predErrors_fused
    
    print("Result generation....")
    AssignValuesByPixelID(input_raster_path=PidRaster, output_raster_path=out_fusion, pid_array=PidArray, value_array=PredsArray)




def finalPrediction_PixelCombination(scaler, df_Path,model,PidRaster,out_fusion,GDEM):
 
    df_pred = pd.read_json(df_Path)
    columns_to_check = ["CTSE",GDEM ]  
    df_pred = df_pred[(df_pred[columns_to_check] > 0).all(axis=1)]
    
    PidArray = np.array(df_pred['Pid'])
    CTSEArray = np.array(df_pred['CTSE'])
    GDEMArray = np.array(df_pred[GDEM])
    DEMArray = np.array(df_pred['DEM'])
    
    df_pred['DEM_CTSE']=df_pred['DEM']-df_pred['CTSE']
    df_pred['DEM_GDEM']=df_pred['DEM']-df_pred[GDEM]

    PredError_CTSE=np.array(df_pred['DEM_CTSE'])
    PredError_GDEM=np.array(df_pred['DEM_GDEM'])
   
    df_pred=df_pred[usingFeatures]

    input_features = np.array(df_pred)
    input_features = scaler.transform(input_features) 
    

    predictions = model.predict(input_features)

   
    predErrors_fused = np.where(predictions == 0, PredError_CTSE, PredError_GDEM)

    PredsArray = DEMArray - predErrors_fused
    
    print("Result generation....")
    AssignValuesByPixelID(input_raster_path=PidRaster, output_raster_path=out_fusion, pid_array=PidArray, value_array=PredsArray)




"""
Classification 0-1 with threshold
"""
def finalPrediction_Prob_threshold(df_Path, model, PidRaster, out_fusion, GDEM):

    df_pred = pd.read_json(df_Path)
    columns_to_check = ["CTSE", GDEM]  
    df_pred = df_pred[(df_pred[columns_to_check] > 0).all(axis=1)]

    PidArray = np.array(df_pred['Pid'])
    DEMArray = np.array(df_pred['DEM'])


    df_pred['DEM_CTSE'] = df_pred['DEM'] - df_pred['CTSE']
    df_pred['DEM_GDEM'] = df_pred['DEM'] - df_pred[GDEM]

    PredError_CTSE = np.array(df_pred['DEM_CTSE'])
    PredError_GDEM = np.array(df_pred['DEM_GDEM'])


    df_pred_input = df_pred[usingFeatures]
    input_features = np.array(df_pred_input)
    input_features = preprocessing.StandardScaler().fit_transform(input_features)


    probs = model.predict_proba(input_features)

 
    condition = np.abs(PredError_CTSE - PredError_GDEM) < 1
    predErrors_fused = np.where(
        condition,
        PredError_CTSE,  
        PredError_CTSE * probs[:, 0] + PredError_GDEM * probs[:, 1] 
    )

    PredsArray = DEMArray - predErrors_fused

    AssignValuesByPixelID(
        input_raster_path=PidRaster,
        output_raster_path=out_fusion,
        pid_array=PidArray,
        value_array=PredsArray
    )



def finalPrediction_RegressionBias(scaler, df_Path, model, PidRaster, out_fusion, GDEM):

    df_pred = pd.read_json(df_Path)
    columns_to_check = ["CTSE", GDEM] 
    df_pred = df_pred[(df_pred[columns_to_check] > 0).all(axis=1)]

    PidArray = np.array(df_pred['Pid'])
    DEMArray = np.array(df_pred['DEM'])


    df_pred['DEM_CTSE'] = df_pred['DEM'] - df_pred['CTSE']
    df_pred['DEM_GDEM'] = df_pred['DEM'] - df_pred[GDEM]



    df_pred_input = df_pred[usingFeatures]
    input_features = np.array(df_pred_input)

    input_features = np.array(df_pred_input)
    input_features = scaler.transform(input_features)


    predErrors = model.predict(input_features)

    PredsArray = DEMArray - predErrors

    AssignValuesByPixelID(
        input_raster_path=PidRaster,
        output_raster_path=out_fusion,
        pid_array=PidArray,
        value_array=PredsArray
    )





def finalPrediction_RegressionCoeff(scaler, df_Path, model, PidRaster, out_fusion, GDEM):

    df_pred = pd.read_json(df_Path)
    columns_to_check = ["CTSE", GDEM] 
    df_pred = df_pred[(df_pred[columns_to_check] > 0).all(axis=1)]

    PidArray = np.array(df_pred['Pid'])
    DEMArray = np.array(df_pred['DEM'])


    df_pred['DEM_CTSE'] = df_pred['DEM'] - df_pred['CTSE']
    df_pred['DEM_GDEM'] = df_pred['DEM'] - df_pred[GDEM]

    PredError_CTSE = np.array(df_pred['DEM_CTSE'])
    PredError_GDEM = np.array(df_pred['DEM_GDEM'])

 
    df_pred_input = df_pred[usingFeatures]
    input_features = np.array(df_pred_input)

    input_features = scaler.transform(input_features)  



    reg_pred = model.predict(input_features)  

    reg_pred = np.clip(reg_pred, a_min=0, a_max=1)

    predErrors_fused=PredError_CTSE * reg_pred + PredError_GDEM * (1-reg_pred)

    PredsArray = DEMArray - predErrors_fused

    AssignValuesByPixelID(
        input_raster_path=PidRaster,
        output_raster_path=out_fusion,
        pid_array=PidArray,
        value_array=PredsArray
    )



# Training

In [ ]:
training_Folder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\AFusion_Validation\TrainingPoints_MR\Interpolation_Full"
k=0
Area_list=[1,2,3,4,5,6,7,8,9,10]
for k, id in enumerate(Area_list):

    try:
        area=f"ICESat_Area{id}"
        print(f"------   {area}  -------------")

        shp_path=os.path.join(training_Folder,f"ATL08Points_Area{id}_training.shp")
        train_data,train_labels,GDEMDict,_=genTrainingData_Classification(shp_path,GDEM="Comp")

        modelPath=os.path.join(r"saveFolder",f"BestFusion_{area}_AllAreas_v2.pkl")
        automl=train_AutoML_Classification(train_data,train_labels, modelPath=modelPath, IsValidation=False,Minute=120)


    except Exception as e:
        print(f"{e}")
 




# Fusion

In [ ]:

training_Folder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\AFusion_Validation\TrainingPoints_MR\Interpolation_Full"
GDEM_list=["Fathom","Fathom","Fathom","FAB","FAB","FAB","GEDTM30","FAB","Fathom","Fathom"]  
Area_list=[1,2,3,4,5,6,7,8,9,10]
FusedGDEM_Folder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\MLCorrectedResults\FusionResults_MR\FullAreas"
ToFusionData_Folder=r"I:\DC_HR\NPZ_ICESatAreas\Fusion_Full"

for k, id in enumerate(Area_list):


    area=f"ICESat_Area{id}"
    print(f"------   {area}  -------------")

    shp_path=os.path.join(training_Folder,f"ATL08Points_Area{id}_training.shp")
    _,_,_, scaler =genTrainingData_RegressionCoeff(shp_path,GDEM="Comp")  

    PidFolder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\Pid2Cop"

    out_fusion=os.path.join(FusedGDEM_Folder,f"RegressionCoeff_TopOther_{area}_AllAreas.tif")
    
    modelPath=os.path.join(r"Models\Fusion_MR",f"RegressionCoeff_TopOther_{area}_AllAreas.pkl") 

    with open(modelPath, "rb") as f:
        automl = pickle.load(f)


    PidRaster=os.path.join(PidFolder,f"Pid_Cop_WGS84_{area}.tif")
    df_Path=os.path.join(ToFusionData_Folder,f"ToFuseData_{area}.json")
    finalPrediction_RegressionCoeff(scaler, df_Path,automl,PidRaster,out_fusion,GDEM="Comp")


# AVG Fusion

In [ ]:
def finalPrediction_AVG(df_Path, PidRaster, out_fusion, GDEM):
   
    columns_to_check = ["CTSE", GDEM]  
    df_pred = df_pred[(df_pred[columns_to_check] > 0).all(axis=1)]

    PidArray = np.array(df_pred['Pid'])
    DEMArray = np.array(df_pred['DEM'])

    df_pred['DEM_CTSE'] = df_pred['DEM'] - df_pred['CTSE']
    df_pred['DEM_GDEM'] = df_pred['DEM'] - df_pred[GDEM]

    PredError_CTSE = np.array(df_pred['DEM_CTSE'])
    PredError_GDEM = np.array(df_pred['DEM_GDEM'])


    predErrors_fused= (PredError_CTSE + PredError_GDEM )/2


    PredsArray = DEMArray - predErrors_fused

    AssignValuesByPixelID(
        input_raster_path=PidRaster,
        output_raster_path=out_fusion,
        pid_array=PidArray,
        value_array=PredsArray
    )


Area_list=[1,2,3,4,5,6,7,8,9,10]
training_Folder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\AFusion_Validation\TrainingPoints_MR\Interpolation_Full"
GDEM_list=["Fathom","Fathom","Fathom","FAB","FAB","FAB","GEDTM30","FAB","Fathom","Fathom"]  
FusedGDEM_Folder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\MLCorrectedResults\FusionResults_MR\FullAreas"
ToFusionData_Folder=r"I:\DC_HR\NPZ_ICESatAreas\Fusion_Full"

k=0
for k, id in enumerate(Area_list):

    area=f"ICESat_Area{id}"
    print(f"------   {area}  -------------")

    PidFolder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\Pid2Cop"
    out_fusion=os.path.join(FusedGDEM_Folder,f"AVGFusion_{area}_GDEM.tif")

    PidRaster=os.path.join(PidFolder,f"Pid_Cop_WGS84_{area}.tif")
    df_Path=os.path.join(ToFusionData_Folder,f"ToFuseData_{area}.json")
    
    finalPrediction_AVG(df_Path,PidRaster,out_fusion,GDEM=GDEM_list[k])

    
 



# Validation-ICESat-2  

In [2]:
import json
import arcpy
from arcpy.sa import *
import numpy as np
import os
import pandas as pd
from osgeo import osr
from osgeo import gdal
import glob

def filterPandas(df):
    
    df['DEM_ATL'] = df['DEM'] - df['EGM2008'] 
    mean_value = df['DEM_ATL'].mean()
    std_value = df['DEM_ATL'].std()
    lower_bound = mean_value - 3 * std_value
    upper_bound = mean_value + 3 * std_value
    filtered_df = df[(df['DEM_ATL'] >= lower_bound) & (df['DEM_ATL'] <= upper_bound)]
    filtered_df = filtered_df.reset_index(drop=True)
    return filtered_df


def run(CopDEM, FusionDEM, CTSEDEM,FABDEM,FathomDEM,GEDTM30,ATL08Points):

    raster_mappings = [
            (CopDEM, "DEM"),
            (FusionDEM, "FusionDEM"),
            (CTSEDEM, "CTSEDEM"),
            (FABDEM, "FABDEM"),
            (FathomDEM, "FathomDEM"),
            (GEDTM30, "GEDTM30")
        ]

    in_rasters_str = ";".join([f"{raster} {field}" for raster, field in raster_mappings])

 
    fields_to_add = [field for _, field in raster_mappings]
    existing_fields = [f.name for f in arcpy.ListFields(ATL08Points)]
    fields_to_delete = [field for field in fields_to_add if field in existing_fields]

    if fields_to_delete:
        arcpy.DeleteField_management(ATL08Points, fields_to_delete)

    arcpy.sa.ExtractMultiValuesToPoints(
        in_point_features=ATL08Points,
        in_rasters=in_rasters_str,
        bilinear_interpolate_values="BILINEAR"
    )

    columns_to_check = ["DEM", "FusionDEM", "CTSEDEM",'FABDEM','FathomDEM','GEDTM30']  

    field_names = [field.name for field in arcpy.ListFields(ATL08Points) if field.type != 'Geometry']

    data = []
    with arcpy.da.SearchCursor(ATL08Points, field_names) as cursor:
        for row in cursor:
            data.append(row)
    df = pd.DataFrame(data, columns=field_names)

    df = df[(df[columns_to_check] > 0).all(axis=1)]

    df=filterPandas(df)
    df = df[~df["LandC"].isin( [187,210,0] )] 

    Cop_RMSE = np.sqrt(np.mean((df['EGM2008'] - df['DEM']) ** 2))
    FusionDEM_RMSE = np.sqrt(np.mean((df['EGM2008'] - df['FusionDEM']) ** 2))
    CTSEDEM_RMSE = np.sqrt(np.mean((df['EGM2008'] - df['CTSEDEM']) ** 2))
    Fathom_RMSE = np.sqrt(np.mean((df['EGM2008'] - df['FathomDEM']) ** 2))
    FABDEM_RMSE = np.sqrt(np.mean((df['EGM2008'] - df['FABDEM']) ** 2))
    GEDTM30_DEM_RMSE = np.sqrt(np.mean((df['EGM2008'] - df['GEDTM30']) ** 2))

    print(f" --RMSE-- Raw: {Cop_RMSE:0.2f}  FusionDEM:{FusionDEM_RMSE:0.2f}  CTSEDEM: {CTSEDEM_RMSE:0.2f}    FathomDEM: {Fathom_RMSE:0.2f}   FAB: {FABDEM_RMSE:0.2f}    GEDTM30: {GEDTM30_DEM_RMSE:0.2f} ")





## ICEsat-2 > 30m

In [ ]:
Area_list=["ICESat_Area"+ str(id) for id in [1,2,3,4,5,6,7,8,9,10]]

In [ ]:

CTSEFolder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\MLCorrectedResults\CorrectedDEM_MR_AllAreas"

sharedFolder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion"
ATL08Folder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\AFusion_Validation\TrainingATL08_v2"

FusedGDEM_Folder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\MLCorrectedResults\FusionResults_MR\FullAreas"  




for k, areaLabel in  enumerate(Area_list) :

    print(f"IA {k+1} is working....")

 
    ATL08Points=os.path.join(ATL08Folder,f"ATL08Points_{areaLabel}_validation.shp")

    Cop_DEM=os.path.join(os.path.join(sharedFolder,"CopDEM"),f"Cop_WGS84_{areaLabel}.tif")
    Cop_DEM=arcpy.Raster(Cop_DEM)
    CTSEDEM=os.path.join(CTSEFolder,f"{areaLabel}_Proposed_ALLAreas.tif")

    CTSEDEM=arcpy.Raster(CTSEDEM)

    FABDEM=os.path.join(os.path.join(sharedFolder,"FABDEM"),f"FAB_WGS84_{areaLabel}.tif")
    FABDEM=arcpy.Raster(FABDEM)

    FathomDEM=os.path.join(os.path.join(sharedFolder,"FathomDEM"),f"Fathom_{areaLabel}.tif")
    FathomDEM=arcpy.Raster(FathomDEM)*0.01

    GEDTM30=os.path.join(os.path.join(sharedFolder,"GEDTM30"),f"GEDTM30_{areaLabel}.tif")
    GEDTM30=arcpy.Raster(GEDTM30)
    
    FusionDEM=os.path.join(FusedGDEM_Folder,f"ClassificationBestFusion_{areaLabel}_AllAreas.tif")  
  
    FusionDEM=arcpy.Raster(FusionDEM)
    run(Cop_DEM, FusionDEM, CTSEDEM,FABDEM,FathomDEM,GEDTM30,ATL08Points)
    print()

 

IA 1 is working....
 --RMSE-- Raw: 15.58  FusionDEM:8.12  CTSEDEM: 8.59    FathomDEM: 8.60   FAB: 9.10    GEDTM30: 9.23 

IA 2 is working....
 --RMSE-- Raw: 13.32  FusionDEM:8.06  CTSEDEM: 8.52    FathomDEM: 9.35   FAB: 9.53    GEDTM30: 9.58 

IA 3 is working....
 --RMSE-- Raw: 14.43  FusionDEM:8.05  CTSEDEM: 8.81    FathomDEM: 9.32   FAB: 10.26    GEDTM30: 10.06 

IA 4 is working....
 --RMSE-- Raw: 21.80  FusionDEM:9.11  CTSEDEM: 9.42    FathomDEM: 11.64   FAB: 10.35    GEDTM30: 10.67 

IA 5 is working....
 --RMSE-- Raw: 25.74  FusionDEM:12.69  CTSEDEM: 13.22    FathomDEM: 14.85   FAB: 14.09    GEDTM30: 15.15 

IA 6 is working....
 --RMSE-- Raw: 22.82  FusionDEM:10.50  CTSEDEM: 10.98    FathomDEM: 12.33   FAB: 11.31    GEDTM30: 12.80 

IA 7 is working....
 --RMSE-- Raw: 21.37  FusionDEM:10.67  CTSEDEM: 10.98    FathomDEM: 12.68   FAB: 12.40    GEDTM30: 12.01 

IA 8 is working....
 --RMSE-- Raw: 19.84  FusionDEM:10.23  CTSEDEM: 10.76    FathomDEM: 12.55   FAB: 11.61    GEDTM30: 11.82 


In [ ]:

CTSEFolder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\MLCorrectedResults\CorrectedDEM_MR_AllAreas"

sharedFolder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion"
ATL08Folder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\AFusion_Validation\TrainingATL08_v2"

FusedGDEM_Folder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\MLCorrectedResults\FusionResults_MR\FullAreas"  


for k, areaLabel in  enumerate(Area_list) :

    print(f"IA {k+1} is working....")

 
    ATL08Points=os.path.join(ATL08Folder,f"ATL08Points_{areaLabel}_validation.shp")

    Cop_DEM=os.path.join(os.path.join(sharedFolder,"CopDEM"),f"Cop_WGS84_{areaLabel}.tif")
    Cop_DEM=arcpy.Raster(Cop_DEM)
    CTSEDEM=os.path.join(CTSEFolder,f"{areaLabel}_Proposed_ALLAreas.tif")

    CTSEDEM=arcpy.Raster(CTSEDEM)

    FABDEM=os.path.join(os.path.join(sharedFolder,"FABDEM"),f"FAB_WGS84_{areaLabel}.tif")
    FABDEM=arcpy.Raster(FABDEM)

    FathomDEM=os.path.join(os.path.join(sharedFolder,"FathomDEM"),f"Fathom_{areaLabel}.tif")
    FathomDEM=arcpy.Raster(FathomDEM)*0.01

    GEDTM30=os.path.join(os.path.join(sharedFolder,"GEDTM30"),f"GEDTM30_{areaLabel}.tif")
    GEDTM30=arcpy.Raster(GEDTM30)
    
    FusionDEM=os.path.join(FusedGDEM_Folder,f"RegressionCoef_{areaLabel}_AllAreas.tif") 

    FusionDEM=arcpy.Raster(FusionDEM)
    run(Cop_DEM, FusionDEM, CTSEDEM,FABDEM,FathomDEM,GEDTM30,ATL08Points)
    print()

 

IA 1 is working....
 --RMSE-- Raw: 15.58  FusionDEM:8.33  CTSEDEM: 8.59    FathomDEM: 8.60   FAB: 9.10    GEDTM30: 9.23 

IA 2 is working....
 --RMSE-- Raw: 13.32  FusionDEM:8.53  CTSEDEM: 8.52    FathomDEM: 9.35   FAB: 9.53    GEDTM30: 9.58 

IA 3 is working....
 --RMSE-- Raw: 14.43  FusionDEM:8.54  CTSEDEM: 8.81    FathomDEM: 9.32   FAB: 10.26    GEDTM30: 10.06 

IA 4 is working....
 --RMSE-- Raw: 21.80  FusionDEM:9.27  CTSEDEM: 9.42    FathomDEM: 11.64   FAB: 10.35    GEDTM30: 10.67 

IA 5 is working....
 --RMSE-- Raw: 25.74  FusionDEM:12.63  CTSEDEM: 13.22    FathomDEM: 14.85   FAB: 14.09    GEDTM30: 15.15 

IA 6 is working....
 --RMSE-- Raw: 22.81  FusionDEM:10.41  CTSEDEM: 10.97    FathomDEM: 12.32   FAB: 11.31    GEDTM30: 12.80 

IA 7 is working....
 --RMSE-- Raw: 21.37  FusionDEM:10.81  CTSEDEM: 10.98    FathomDEM: 12.68   FAB: 12.40    GEDTM30: 12.01 

IA 8 is working....
 --RMSE-- Raw: 19.84  FusionDEM:10.58  CTSEDEM: 10.76    FathomDEM: 12.55   FAB: 11.61    GEDTM30: 11.82 


In [ ]:

CTSEFolder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\MLCorrectedResults\CorrectedDEM_MR_AllAreas"

sharedFolder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion"
ATL08Folder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\AFusion_Validation\TrainingATL08_v2"

FusedGDEM_Folder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\MLCorrectedResults\FusionResults_MR\FullAreas"  


for k, areaLabel in  enumerate(Area_list) :

    print(f"IA {k+1} is working....")

 
    ATL08Points=os.path.join(ATL08Folder,f"ATL08Points_{areaLabel}_validation.shp")

    Cop_DEM=os.path.join(os.path.join(sharedFolder,"CopDEM"),f"Cop_WGS84_{areaLabel}.tif")
    Cop_DEM=arcpy.Raster(Cop_DEM)
    CTSEDEM=os.path.join(CTSEFolder,f"{areaLabel}_Proposed_ALLAreas.tif")

    CTSEDEM=arcpy.Raster(CTSEDEM)

    FABDEM=os.path.join(os.path.join(sharedFolder,"FABDEM"),f"FAB_WGS84_{areaLabel}.tif")
    FABDEM=arcpy.Raster(FABDEM)

    FathomDEM=os.path.join(os.path.join(sharedFolder,"FathomDEM"),f"Fathom_{areaLabel}.tif")
    FathomDEM=arcpy.Raster(FathomDEM)*0.01

    GEDTM30=os.path.join(os.path.join(sharedFolder,"GEDTM30"),f"GEDTM30_{areaLabel}.tif")
    GEDTM30=arcpy.Raster(GEDTM30)
    
    FusionDEM=os.path.join(FusedGDEM_Folder,f"Classification_GDTMTopOther_{areaLabel}_AllAreas.tif")   


    FusionDEM=arcpy.Raster(FusionDEM)
    run(Cop_DEM, FusionDEM, CTSEDEM,FABDEM,FathomDEM,GEDTM30,ATL08Points)
    print()

 

IA 1 is working....
 --RMSE-- Raw: 15.58  FusionDEM:8.20  CTSEDEM: 8.59    FathomDEM: 8.60   FAB: 9.10    GEDTM30: 9.23 

IA 2 is working....
 --RMSE-- Raw: 13.32  FusionDEM:8.27  CTSEDEM: 8.52    FathomDEM: 9.35   FAB: 9.53    GEDTM30: 9.58 

IA 3 is working....
 --RMSE-- Raw: 14.43  FusionDEM:8.44  CTSEDEM: 8.81    FathomDEM: 9.32   FAB: 10.26    GEDTM30: 10.06 

IA 4 is working....
 --RMSE-- Raw: 21.80  FusionDEM:9.29  CTSEDEM: 9.42    FathomDEM: 11.64   FAB: 10.35    GEDTM30: 10.67 

IA 5 is working....
 --RMSE-- Raw: 25.74  FusionDEM:12.59  CTSEDEM: 13.22    FathomDEM: 14.85   FAB: 14.09    GEDTM30: 15.15 

IA 6 is working....
 --RMSE-- Raw: 22.82  FusionDEM:10.16  CTSEDEM: 10.98    FathomDEM: 12.33   FAB: 11.31    GEDTM30: 12.80 

IA 7 is working....
 --RMSE-- Raw: 21.37  FusionDEM:10.60  CTSEDEM: 10.98    FathomDEM: 12.68   FAB: 12.40    GEDTM30: 12.01 

IA 8 is working....
 --RMSE-- Raw: 19.84  FusionDEM:10.61  CTSEDEM: 10.76    FathomDEM: 12.55   FAB: 11.61    GEDTM30: 11.82 


In [ ]:

CTSEFolder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\MLCorrectedResults\CorrectedDEM_MR_AllAreas"

sharedFolder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion"
ATL08Folder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\AFusion_Validation\TrainingATL08_v2"

FusedGDEM_Folder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\MLCorrectedResults\FusionResults_MR\FullAreas"  


for k, areaLabel in  enumerate(Area_list) :

    print(f"IA {k+1} is working....")

 
    ATL08Points=os.path.join(ATL08Folder,f"ATL08Points_{areaLabel}_validation.shp")

    Cop_DEM=os.path.join(os.path.join(sharedFolder,"CopDEM"),f"Cop_WGS84_{areaLabel}.tif")
    Cop_DEM=arcpy.Raster(Cop_DEM)
    CTSEDEM=os.path.join(CTSEFolder,f"{areaLabel}_Proposed_ALLAreas.tif")

    CTSEDEM=arcpy.Raster(CTSEDEM)

    FABDEM=os.path.join(os.path.join(sharedFolder,"FABDEM"),f"FAB_WGS84_{areaLabel}.tif")
    FABDEM=arcpy.Raster(FABDEM)

    FathomDEM=os.path.join(os.path.join(sharedFolder,"FathomDEM"),f"Fathom_{areaLabel}.tif")
    FathomDEM=arcpy.Raster(FathomDEM)*0.01

    GEDTM30=os.path.join(os.path.join(sharedFolder,"GEDTM30"),f"GEDTM30_{areaLabel}.tif")
    GEDTM30=arcpy.Raster(GEDTM30)
    

    FusionDEM=os.path.join(FusedGDEM_Folder,f"RegressionCoeff_TopOther_{areaLabel}_AllAreas.tif")


    FusionDEM=arcpy.Raster(FusionDEM)
    run(Cop_DEM, FusionDEM, CTSEDEM,FABDEM,FathomDEM,GEDTM30,ATL08Points)
    print()

 

IA 1 is working....
 --RMSE-- Raw: 15.58  FusionDEM:8.25  CTSEDEM: 8.59    FathomDEM: 8.60   FAB: 9.10    GEDTM30: 9.23 

IA 2 is working....
 --RMSE-- Raw: 13.32  FusionDEM:8.25  CTSEDEM: 8.52    FathomDEM: 9.35   FAB: 9.53    GEDTM30: 9.58 

IA 3 is working....
 --RMSE-- Raw: 14.43  FusionDEM:8.33  CTSEDEM: 8.81    FathomDEM: 9.32   FAB: 10.26    GEDTM30: 10.06 

IA 4 is working....
 --RMSE-- Raw: 21.80  FusionDEM:9.18  CTSEDEM: 9.42    FathomDEM: 11.64   FAB: 10.35    GEDTM30: 10.67 

IA 5 is working....
 --RMSE-- Raw: 25.74  FusionDEM:12.89  CTSEDEM: 13.22    FathomDEM: 14.85   FAB: 14.09    GEDTM30: 15.15 

IA 6 is working....
 --RMSE-- Raw: 22.82  FusionDEM:10.64  CTSEDEM: 10.98    FathomDEM: 12.33   FAB: 11.31    GEDTM30: 12.80 

IA 7 is working....
 --RMSE-- Raw: 21.37  FusionDEM:10.77  CTSEDEM: 10.98    FathomDEM: 12.68   FAB: 12.40    GEDTM30: 12.01 

IA 8 is working....
 --RMSE-- Raw: 19.84  FusionDEM:10.38  CTSEDEM: 10.76    FathomDEM: 12.55   FAB: 11.61    GEDTM30: 11.82 


In [ ]:

for k, areaLabel in  enumerate(Area_list) :

    print(f"IA {k+1} is working....")

 
    ATL08Points=os.path.join(ATL08Folder,f"ATL08Points_{areaLabel}_validation.shp")

    Cop_DEM=os.path.join(os.path.join(sharedFolder,"CopDEM"),f"Cop_WGS84_{areaLabel}.tif")
    Cop_DEM=arcpy.Raster(Cop_DEM)
    CTSEDEM=os.path.join(CTSEFolder,f"{areaLabel}_Proposed_ALLAreas.tif")

    CTSEDEM=arcpy.Raster(CTSEDEM)

    FABDEM=os.path.join(os.path.join(sharedFolder,"FABDEM"),f"FAB_WGS84_{areaLabel}.tif")
    FABDEM=arcpy.Raster(FABDEM)

    FathomDEM=os.path.join(os.path.join(sharedFolder,"FathomDEM"),f"Fathom_{areaLabel}.tif")
    FathomDEM=arcpy.Raster(FathomDEM)*0.01

    GEDTM30=os.path.join(os.path.join(sharedFolder,"GEDTM30"),f"GEDTM30_{areaLabel}.tif")
    GEDTM30=arcpy.Raster(GEDTM30)
    
    FusionDEM=os.path.join(FusedGDEM_Folder,f"AVGFusion_{areaLabel}_Comp.tif")

    FusionDEM=arcpy.Raster(FusionDEM)
    run(Cop_DEM, FusionDEM, CTSEDEM,FABDEM,FathomDEM,GEDTM30,ATL08Points)
    print()

 

IA 1 is working....
 --RMSE-- Raw: 15.58  FusionDEM:8.31  CTSEDEM: 8.59    FathomDEM: 8.60   FAB: 9.10    GEDTM30: 9.23 

IA 2 is working....
 --RMSE-- Raw: 13.32  FusionDEM:8.38  CTSEDEM: 8.52    FathomDEM: 9.35   FAB: 9.53    GEDTM30: 9.58 

IA 3 is working....
 --RMSE-- Raw: 14.43  FusionDEM:8.60  CTSEDEM: 8.81    FathomDEM: 9.32   FAB: 10.26    GEDTM30: 10.06 

IA 4 is working....
 --RMSE-- Raw: 21.80  FusionDEM:9.23  CTSEDEM: 9.42    FathomDEM: 11.64   FAB: 10.35    GEDTM30: 10.67 

IA 5 is working....
 --RMSE-- Raw: 25.74  FusionDEM:13.03  CTSEDEM: 13.22    FathomDEM: 14.85   FAB: 14.09    GEDTM30: 15.15 

IA 6 is working....
 --RMSE-- Raw: 22.82  FusionDEM:10.69  CTSEDEM: 10.98    FathomDEM: 12.33   FAB: 11.31    GEDTM30: 12.80 

IA 7 is working....
 --RMSE-- Raw: 21.37  FusionDEM:10.82  CTSEDEM: 10.98    FathomDEM: 12.68   FAB: 12.40    GEDTM30: 12.01 

IA 8 is working....
 --RMSE-- Raw: 19.84  FusionDEM:10.49  CTSEDEM: 10.76    FathomDEM: 12.55   FAB: 11.61    GEDTM30: 11.82 


In [ ]:
for k, areaLabel in  enumerate(Area_list) :

    print(f"IA {k+1} is working....")

 
    ATL08Points=os.path.join(ATL08Folder,f"ATL08Points_{areaLabel}_validation.shp")

    Cop_DEM=os.path.join(os.path.join(sharedFolder,"CopDEM"),f"Cop_WGS84_{areaLabel}.tif")
    Cop_DEM=arcpy.Raster(Cop_DEM)
    CTSEDEM=os.path.join(CTSEFolder,f"{areaLabel}_Proposed_ALLAreas.tif")

    CTSEDEM=arcpy.Raster(CTSEDEM)

    FABDEM=os.path.join(os.path.join(sharedFolder,"FABDEM"),f"FAB_WGS84_{areaLabel}.tif")
    FABDEM=arcpy.Raster(FABDEM)

    FathomDEM=os.path.join(os.path.join(sharedFolder,"FathomDEM"),f"Fathom_{areaLabel}.tif")
    FathomDEM=arcpy.Raster(FathomDEM)*0.01

    GEDTM30=os.path.join(os.path.join(sharedFolder,"GEDTM30"),f"GEDTM30_{areaLabel}.tif")
    GEDTM30=arcpy.Raster(GEDTM30)
     
    FusionDEM=os.path.join(FusedGDEM_Folder,f"AVGFusion_{areaLabel}_GDEM.tif")  


    FusionDEM=arcpy.Raster(FusionDEM)

    run(Cop_DEM, FusionDEM, CTSEDEM,FABDEM,FathomDEM,GEDTM30,ATL08Points)

    print()

 



IA 1 is working....
 --RMSE-- Raw: 15.58  FusionDEM:8.41  CTSEDEM: 8.59    FathomDEM: 8.60   FAB: 9.10    GEDTM30: 9.23 

IA 2 is working....
 --RMSE-- Raw: 13.32  FusionDEM:8.80  CTSEDEM: 8.52    FathomDEM: 9.35   FAB: 9.53    GEDTM30: 9.58 

IA 3 is working....
 --RMSE-- Raw: 14.43  FusionDEM:8.83  CTSEDEM: 8.81    FathomDEM: 9.32   FAB: 10.26    GEDTM30: 10.06 

IA 4 is working....
 --RMSE-- Raw: 21.80  FusionDEM:9.50  CTSEDEM: 9.42    FathomDEM: 11.64   FAB: 10.35    GEDTM30: 10.67 

IA 5 is working....
 --RMSE-- Raw: 25.74  FusionDEM:13.07  CTSEDEM: 13.22    FathomDEM: 14.85   FAB: 14.09    GEDTM30: 15.15 

IA 6 is working....
 --RMSE-- Raw: 22.81  FusionDEM:10.68  CTSEDEM: 10.97    FathomDEM: 12.32   FAB: 11.31    GEDTM30: 12.80 

IA 7 is working....
 --RMSE-- Raw: 21.37  FusionDEM:11.05  CTSEDEM: 10.98    FathomDEM: 12.68   FAB: 12.40    GEDTM30: 12.01 

IA 8 is working....
 --RMSE-- Raw: 19.84  FusionDEM:10.83  CTSEDEM: 10.76    FathomDEM: 12.55   FAB: 11.61    GEDTM30: 11.82 


In [ ]:
for k, areaLabel in  enumerate(Area_list) :

    print(f"IA {k+1} is working....")

 
    ATL08Points=os.path.join(ATL08Folder,f"ATL08Points_{areaLabel}_validation.shp")

    Cop_DEM=os.path.join(os.path.join(sharedFolder,"CopDEM"),f"Cop_WGS84_{areaLabel}.tif")
    Cop_DEM=arcpy.Raster(Cop_DEM)
    CTSEDEM=os.path.join(CTSEFolder,f"{areaLabel}_Proposed_ALLAreas.tif")

    CTSEDEM=arcpy.Raster(CTSEDEM)

    FABDEM=os.path.join(os.path.join(sharedFolder,"FABDEM"),f"FAB_WGS84_{areaLabel}.tif")
    FABDEM=arcpy.Raster(FABDEM)

    FathomDEM=os.path.join(os.path.join(sharedFolder,"FathomDEM"),f"Fathom_{areaLabel}.tif")
    FathomDEM=arcpy.Raster(FathomDEM)*0.01

    GEDTM30=os.path.join(os.path.join(sharedFolder,"GEDTM30"),f"GEDTM30_{areaLabel}.tif")
    GEDTM30=arcpy.Raster(GEDTM30)
     
    FusionDEM=os.path.join(FusedGDEM_Folder,f"AVGFusion_{areaLabel}_Comp.tif")  


    FusionDEM=arcpy.Raster(FusionDEM)

    run(Cop_DEM, FusionDEM, CTSEDEM,FABDEM,FathomDEM,GEDTM30,ATL08Points)

    print()

 



IA 1 is working....
 --RMSE-- Raw: 15.58  FusionDEM:8.31  CTSEDEM: 8.59    FathomDEM: 8.60   FAB: 9.10    GEDTM30: 9.23 

IA 2 is working....
 --RMSE-- Raw: 13.32  FusionDEM:8.38  CTSEDEM: 8.52    FathomDEM: 9.35   FAB: 9.53    GEDTM30: 9.58 

IA 3 is working....
 --RMSE-- Raw: 14.43  FusionDEM:8.60  CTSEDEM: 8.81    FathomDEM: 9.32   FAB: 10.26    GEDTM30: 10.06 

IA 4 is working....
 --RMSE-- Raw: 21.80  FusionDEM:9.23  CTSEDEM: 9.42    FathomDEM: 11.64   FAB: 10.35    GEDTM30: 10.67 

IA 5 is working....
 --RMSE-- Raw: 25.74  FusionDEM:13.03  CTSEDEM: 13.22    FathomDEM: 14.85   FAB: 14.09    GEDTM30: 15.15 

IA 6 is working....
 --RMSE-- Raw: 22.82  FusionDEM:10.69  CTSEDEM: 10.98    FathomDEM: 12.33   FAB: 11.31    GEDTM30: 12.80 

IA 7 is working....
 --RMSE-- Raw: 21.37  FusionDEM:10.82  CTSEDEM: 10.98    FathomDEM: 12.68   FAB: 12.40    GEDTM30: 12.01 

IA 8 is working....
 --RMSE-- Raw: 19.84  FusionDEM:10.49  CTSEDEM: 10.76    FathomDEM: 12.55   FAB: 11.61    GEDTM30: 11.82 


## Within Different ranges

In [ ]:
import json
import arcpy
from arcpy.sa import *
import numpy as np
import os
import pandas as pd
from osgeo import osr
from osgeo import gdal
import glob

def filterPandas(df):
    
    df['DEM_ATL'] = df['DEM'] - df['EGM2008'] 
    mean_value = df['DEM_ATL'].mean()
    std_value = df['DEM_ATL'].std()
    lower_bound = mean_value - 3 * std_value
    upper_bound = mean_value + 3 * std_value
    filtered_df = df[(df['DEM_ATL'] >= lower_bound) & (df['DEM_ATL'] <= upper_bound)]
    filtered_df = filtered_df.reset_index(drop=True)
    return filtered_df


def runDistance(CopDEM, FusionDEM, CTSEDEM,ATL08Points):

    raster_mappings = [
            (CopDEM, "DEM"),
            (FusionDEM, "FusionDEM"),
            (CTSEDEM, "CTSEDEM"),
    
        ]
    in_rasters_str = ";".join([f"{raster} {field}" for raster, field in raster_mappings])

    fields_to_add = [field for _, field in raster_mappings]
    existing_fields = [f.name for f in arcpy.ListFields(ATL08Points)]
    fields_to_delete = [field for field in fields_to_add if field in existing_fields]

    if fields_to_delete:
        arcpy.DeleteField_management(ATL08Points, fields_to_delete)

    arcpy.sa.ExtractMultiValuesToPoints(
        in_point_features=ATL08Points,
        in_rasters=in_rasters_str,
        bilinear_interpolate_values="BILINEAR"
    )

    columns_to_check = ["DEM", "FusionDEM", "CTSEDEM"]   

    field_names = [field.name for field in arcpy.ListFields(ATL08Points) if field.type != 'Geometry']

    data = []
    with arcpy.da.SearchCursor(ATL08Points, field_names) as cursor:
        for row in cursor:
            data.append(row)
    df = pd.DataFrame(data, columns=field_names)

    df = df[(df[columns_to_check] > 0).all(axis=1)]

    df=filterPandas(df)

    Cop_RMSE = np.sqrt(np.mean((df['EGM2008'] - df['DEM']) ** 2))
    FusionDEM_RMSE = np.sqrt(np.mean((df['EGM2008'] - df['FusionDEM']) ** 2))
    CTSEDEM_RMSE = np.sqrt(np.mean((df['EGM2008'] - df['CTSEDEM']) ** 2))

    imp_Fusion=100*(Cop_RMSE-FusionDEM_RMSE)/Cop_RMSE
    imp_CTSE=100*(Cop_RMSE-CTSEDEM_RMSE)/Cop_RMSE

    print(f" PointsNumber:{len(df)}        Imp_Fusin_CTSE:{imp_Fusion-imp_CTSE:0.2f}%      --RMSE--    Raw: {Cop_RMSE:0.2f}   imp_CTSE: {imp_CTSE:0.2f} ")

    return imp_Fusion-imp_CTSE




In [ ]:
CTDEMFolder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\MLCorrectedResults\CorrectedDEM_MR_AllAreas"
sharedFolder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion"
ATL08Folder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\AFusion_Validation\ValidationPoints_DistanceRanges_v2"

FusedGDEM_Folder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\MLCorrectedResults\FusionResults_MR\FullAreas"  


for rg  in ["30_150", "100_300","300_600", "600_900","900_"]:


    print(f"--------------------{rg}---------------------------")
    imp_list=[]


    for k, areaLabel in  enumerate(Area_list) :

        print(f"IA {k+1} is working....")

    
        ATL08Points=os.path.join(ATL08Folder,f"{areaLabel}_{rg}.shp")

        Cop_DEM=os.path.join(os.path.join(sharedFolder,"CopDEM"),f"Cop_WGS84_{areaLabel}.tif")
        Cop_DEM=arcpy.Raster(Cop_DEM)

        CTSEDEM=os.path.join(CTDEMFolder,f"{areaLabel}_Proposed_AllAreas.tif")  
        CTSEDEM=arcpy.Raster(CTSEDEM)

        FusionDEM=os.path.join(FusedGDEM_Folder,f"ClassificationBestFusion_{areaLabel}_AllAreas.tif")  

        try:
            imp=runDistance(Cop_DEM, FusionDEM, CTSEDEM,ATL08Points)
            imp_list.append(imp)

        except Exception as e:
            print(e)

    

    print(f" RG: {rg}   AVGImp:{sum(imp_list)/len(imp_list):0.2f}  ")
    print()


--------------------30_150---------------------------
IA 1 is working....
 PointsNumber:34196        Imp_Fusin_CTSE:3.47%      --RMSE--    Raw: 13.98   imp_CTSE: 41.60 
IA 2 is working....
 PointsNumber:90109        Imp_Fusin_CTSE:3.52%      --RMSE--    Raw: 13.65   imp_CTSE: 36.15 
IA 3 is working....
 PointsNumber:41243        Imp_Fusin_CTSE:5.34%      --RMSE--    Raw: 14.03   imp_CTSE: 40.36 
IA 4 is working....
 PointsNumber:585        Imp_Fusin_CTSE:1.04%      --RMSE--    Raw: 23.81   imp_CTSE: 53.62 
IA 5 is working....
 PointsNumber:16822        Imp_Fusin_CTSE:2.13%      --RMSE--    Raw: 25.38   imp_CTSE: 49.35 
IA 6 is working....
 PointsNumber:3494        Imp_Fusin_CTSE:2.04%      --RMSE--    Raw: 24.57   imp_CTSE: 52.82 
IA 7 is working....
 PointsNumber:7368        Imp_Fusin_CTSE:1.44%      --RMSE--    Raw: 21.84   imp_CTSE: 45.66 
IA 8 is working....
 PointsNumber:1426        Imp_Fusin_CTSE:2.67%      --RMSE--    Raw: 16.29   imp_CTSE: 46.51 
IA 9 is working....
 PointsNumb

# Validation-GEDI

In [ ]:
import json
import arcpy
from arcpy.sa import *
import numpy as np
import os
import pandas as pd
from osgeo import osr
from osgeo import gdal
import glob

def filterPandas(df):
    
    df['DEM_ATL'] = df['DEM'] - df['EGM2008'] 
    mean_value = df['DEM_ATL'].mean()
    std_value = df['DEM_ATL'].std()
    lower_bound = mean_value - 3 * std_value
    upper_bound = mean_value + 3 * std_value
    filtered_df = df[(df['DEM_ATL'] >= lower_bound) & (df['DEM_ATL'] <= upper_bound)]
    filtered_df = filtered_df.reset_index(drop=True)
    return filtered_df


def run(CopDEM, FusionDEM, CTSEDEM,FABDEM,FathomDEM,GEDTM30,ATL08Points):

    raster_mappings = [
            (CopDEM, "DEM"),
            (FusionDEM, "FusionDEM"),
            (CTSEDEM, "CTSEDEM"),
            (FABDEM, "FABDEM"),
            (FathomDEM, "FathomDEM"),
            (GEDTM30, "GEDTM30")
        ]

    in_rasters_str = ";".join([f"{raster} {field}" for raster, field in raster_mappings])

    fields_to_add = [field for _, field in raster_mappings]
    existing_fields = [f.name for f in arcpy.ListFields(ATL08Points)]
    fields_to_delete = [field for field in fields_to_add if field in existing_fields]

    if fields_to_delete:
        arcpy.DeleteField_management(ATL08Points, fields_to_delete)

    arcpy.sa.ExtractMultiValuesToPoints(
        in_point_features=ATL08Points,
        in_rasters=in_rasters_str,
        bilinear_interpolate_values="BILINEAR"
    )

    columns_to_check = ["DEM", "FusionDEM", "CTSEDEM",'FABDEM','FathomDEM','GEDTM30'] 

    field_names = [field.name for field in arcpy.ListFields(ATL08Points) if field.type != 'Geometry']

    data = []
    with arcpy.da.SearchCursor(ATL08Points, field_names) as cursor:
        for row in cursor:
            data.append(row)
    df = pd.DataFrame(data, columns=field_names)

    df = df[(df[columns_to_check] > 0).all(axis=1)]

    df=filterPandas(df)
    df = df[~df["LandC"].isin( [187,210,0] )] 

    Cop_RMSE = np.sqrt(np.mean((df['EGM2008'] - df['DEM']) ** 2))
    FusionDEM_RMSE = np.sqrt(np.mean((df['EGM2008'] - df['FusionDEM']) ** 2))
    CTSEDEM_RMSE = np.sqrt(np.mean((df['EGM2008'] - df['CTSEDEM']) ** 2))
    Fathom_RMSE = np.sqrt(np.mean((df['EGM2008'] - df['FathomDEM']) ** 2))
    FABDEM_RMSE = np.sqrt(np.mean((df['EGM2008'] - df['FABDEM']) ** 2))
    GEDTM30_DEM_RMSE = np.sqrt(np.mean((df['EGM2008'] - df['GEDTM30']) ** 2))

    print(f" --RMSE-- Raw: {Cop_RMSE:0.2f}  FusionDEM:{FusionDEM_RMSE:0.2f}  CTSEDEM: {CTSEDEM_RMSE:0.2f}    FathomDEM: {Fathom_RMSE:0.2f}   FAB: {FABDEM_RMSE:0.2f}    GEDTM30: {GEDTM30_DEM_RMSE:0.2f} ")




 

In [ ]:
Area_list=["ICESat_Area"+ str(id) for id in [1,2,3,4,5,6,7,8,9,10]]

In [ ]:



CTSEFolder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\MLCorrectedResults\CorrectedDEM_MR_AllAreas"
sharedFolder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion"
ATL08Folder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\AFusion_Validation\GEDI"
FusedGDEM_Folder=r"D:\AAA_Working\DC_HR_Dataset\DEM_Fusion\MLCorrectedResults\FusionResults_MR\FullAreas"  


for k, areaLabel in  enumerate(Area_list) :

    print(f"IA {k+1} is working....")

    ATL08Points=os.path.join(ATL08Folder,f"GEDI_{areaLabel}_Distance30.shp")


    Cop_DEM=os.path.join(os.path.join(sharedFolder,"CopDEM"),f"Cop_WGS84_{areaLabel}.tif")
    Cop_DEM=arcpy.Raster(Cop_DEM)

    CTSEDEM=os.path.join(CTSEFolder,f"{areaLabel}_Proposed_AllAreas.tif")
    CTSEDEM=arcpy.Raster(CTSEDEM)

    FABDEM=os.path.join(os.path.join(sharedFolder,"FABDEM"),f"FAB_WGS84_{areaLabel}.tif")
    FABDEM=arcpy.Raster(FABDEM)

    FathomDEM=os.path.join(os.path.join(sharedFolder,"FathomDEM"),f"Fathom_{areaLabel}.tif")
    FathomDEM=arcpy.Raster(FathomDEM)*0.01

    GEDTM30=os.path.join(os.path.join(sharedFolder,"GEDTM30"),f"GEDTM30_{areaLabel}.tif")
    GEDTM30=arcpy.Raster(GEDTM30)


    FusionDEM=os.path.join(FusedGDEM_Folder,f"ClassificationFusion_{areaLabel}_AllAreas.tif")  

    
    FusionDEM=arcpy.Raster(FusionDEM)

    run(Cop_DEM, FusionDEM, CTSEDEM,FABDEM,FathomDEM,GEDTM30,ATL08Points)

    print()


IA 1 is working....
 --RMSE-- Raw: 17.90  FusionDEM:10.19  CTSEDEM: 10.41    FathomDEM: 10.76   FAB: 11.17    GEDTM30: 11.62 

IA 2 is working....
 --RMSE-- Raw: 15.98  FusionDEM:10.67  CTSEDEM: 10.77    FathomDEM: 11.64   FAB: 12.05    GEDTM30: 12.19 

IA 3 is working....
 --RMSE-- Raw: 16.56  FusionDEM:10.26  CTSEDEM: 10.36    FathomDEM: 11.20   FAB: 12.10    GEDTM30: 12.18 

IA 4 is working....
 --RMSE-- Raw: 23.46  FusionDEM:11.11  CTSEDEM: 11.16    FathomDEM: 13.32   FAB: 12.37    GEDTM30: 13.03 

IA 5 is working....
 --RMSE-- Raw: 26.70  FusionDEM:15.62  CTSEDEM: 15.62    FathomDEM: 17.04   FAB: 17.47    GEDTM30: 18.20 

IA 6 is working....
 --RMSE-- Raw: 27.11  FusionDEM:14.72  CTSEDEM: 14.88    FathomDEM: 16.31   FAB: 15.82    GEDTM30: 17.46 

IA 7 is working....
 --RMSE-- Raw: 20.88  FusionDEM:11.47  CTSEDEM: 11.54    FathomDEM: 12.78   FAB: 13.87    GEDTM30: 12.71 

IA 8 is working....
 --RMSE-- Raw: 22.07  FusionDEM:13.96  CTSEDEM: 14.10    FathomDEM: 14.79   FAB: 15.20    G

In [ ]:
for k, areaLabel in  enumerate(Area_list) :

    print(f"IA {k+1} is working....")

    ATL08Points=os.path.join(ATL08Folder,f"GEDI_{areaLabel}_Distance30.shp")

    Cop_DEM=os.path.join(os.path.join(sharedFolder,"CopDEM"),f"Cop_WGS84_{areaLabel}.tif")
    Cop_DEM=arcpy.Raster(Cop_DEM)

    CTSEDEM=os.path.join(CTSEFolder,f"{areaLabel}_Proposed_AllAreas.tif")
    CTSEDEM=arcpy.Raster(CTSEDEM)

    FABDEM=os.path.join(os.path.join(sharedFolder,"FABDEM"),f"FAB_WGS84_{areaLabel}.tif")
    FABDEM=arcpy.Raster(FABDEM)

    FathomDEM=os.path.join(os.path.join(sharedFolder,"FathomDEM"),f"Fathom_{areaLabel}.tif")
    FathomDEM=arcpy.Raster(FathomDEM)*0.01

    GEDTM30=os.path.join(os.path.join(sharedFolder,"GEDTM30"),f"GEDTM30_{areaLabel}.tif")
    GEDTM30=arcpy.Raster(GEDTM30)

    FusionDEM=os.path.join(FusedGDEM_Folder,f"ClassificationBestFusion_{areaLabel}_AllAreas.tif")  
    
    FusionDEM=arcpy.Raster(FusionDEM)

    run(Cop_DEM, FusionDEM, CTSEDEM,FABDEM,FathomDEM,GEDTM30,ATL08Points)

    print()


IA 1 is working....
 --RMSE-- Raw: 17.90  FusionDEM:10.08  CTSEDEM: 10.41    FathomDEM: 10.76   FAB: 11.17    GEDTM30: 11.62 

IA 2 is working....
 --RMSE-- Raw: 15.98  FusionDEM:10.48  CTSEDEM: 10.77    FathomDEM: 11.64   FAB: 12.05    GEDTM30: 12.19 

IA 3 is working....
 --RMSE-- Raw: 16.56  FusionDEM:10.27  CTSEDEM: 10.36    FathomDEM: 11.20   FAB: 12.10    GEDTM30: 12.18 

IA 4 is working....
 --RMSE-- Raw: 23.46  FusionDEM:11.04  CTSEDEM: 11.16    FathomDEM: 13.32   FAB: 12.37    GEDTM30: 13.03 

IA 5 is working....
 --RMSE-- Raw: 26.70  FusionDEM:15.43  CTSEDEM: 15.62    FathomDEM: 17.04   FAB: 17.47    GEDTM30: 18.20 

IA 6 is working....
 --RMSE-- Raw: 27.12  FusionDEM:14.57  CTSEDEM: 14.88    FathomDEM: 16.31   FAB: 15.82    GEDTM30: 17.46 

IA 7 is working....
 --RMSE-- Raw: 20.88  FusionDEM:11.47  CTSEDEM: 11.54    FathomDEM: 12.78   FAB: 13.87    GEDTM30: 12.71 

IA 8 is working....
 --RMSE-- Raw: 22.07  FusionDEM:13.98  CTSEDEM: 14.10    FathomDEM: 14.79   FAB: 15.20    G

In [ ]:
for k, areaLabel in  enumerate(Area_list) :

    print(f"IA {k+1} is working....")

    ATL08Points=os.path.join(ATL08Folder,f"GEDI_{areaLabel}_Distance30.shp")

    Cop_DEM=os.path.join(os.path.join(sharedFolder,"CopDEM"),f"Cop_WGS84_{areaLabel}.tif")
    Cop_DEM=arcpy.Raster(Cop_DEM)

    CTSEDEM=os.path.join(CTSEFolder,f"{areaLabel}_Proposed_AllAreas.tif")
    CTSEDEM=arcpy.Raster(CTSEDEM)

    FABDEM=os.path.join(os.path.join(sharedFolder,"FABDEM"),f"FAB_WGS84_{areaLabel}.tif")
    FABDEM=arcpy.Raster(FABDEM)

    FathomDEM=os.path.join(os.path.join(sharedFolder,"FathomDEM"),f"Fathom_{areaLabel}.tif")
    FathomDEM=arcpy.Raster(FathomDEM)*0.01

    GEDTM30=os.path.join(os.path.join(sharedFolder,"GEDTM30"),f"GEDTM30_{areaLabel}.tif")
    GEDTM30=arcpy.Raster(GEDTM30)


    FusionDEM=os.path.join(FusedGDEM_Folder,f"RegressionCoef_{areaLabel}_AllAreas.tif") 
    
    FusionDEM=arcpy.Raster(FusionDEM)

    run(Cop_DEM, FusionDEM, CTSEDEM,FABDEM,FathomDEM,GEDTM30,ATL08Points)

    print()


IA 1 is working....
 --RMSE-- Raw: 17.90  FusionDEM:10.31  CTSEDEM: 10.41    FathomDEM: 10.76   FAB: 11.17    GEDTM30: 11.62 

IA 2 is working....
 --RMSE-- Raw: 15.98  FusionDEM:10.86  CTSEDEM: 10.77    FathomDEM: 11.64   FAB: 12.05    GEDTM30: 12.19 

IA 3 is working....
 --RMSE-- Raw: 16.56  FusionDEM:10.33  CTSEDEM: 10.36    FathomDEM: 11.20   FAB: 12.10    GEDTM30: 12.18 

IA 4 is working....
 --RMSE-- Raw: 23.46  FusionDEM:11.16  CTSEDEM: 11.16    FathomDEM: 13.32   FAB: 12.37    GEDTM30: 13.03 

IA 5 is working....
 --RMSE-- Raw: 26.70  FusionDEM:15.65  CTSEDEM: 15.62    FathomDEM: 17.04   FAB: 17.47    GEDTM30: 18.20 

IA 6 is working....
 --RMSE-- Raw: 27.11  FusionDEM:14.80  CTSEDEM: 14.88    FathomDEM: 16.31   FAB: 15.82    GEDTM30: 17.46 

IA 7 is working....
 --RMSE-- Raw: 20.88  FusionDEM:11.53  CTSEDEM: 11.54    FathomDEM: 12.78   FAB: 13.87    GEDTM30: 12.71 

IA 8 is working....
 --RMSE-- Raw: 22.07  FusionDEM:14.00  CTSEDEM: 14.10    FathomDEM: 14.79   FAB: 15.20    G

In [ ]:
for k, areaLabel in  enumerate(Area_list) :

    print(f"IA {k+1} is working....")

    ATL08Points=os.path.join(ATL08Folder,f"GEDI_{areaLabel}_Distance30.shp")

    Cop_DEM=os.path.join(os.path.join(sharedFolder,"CopDEM"),f"Cop_WGS84_{areaLabel}.tif")
    Cop_DEM=arcpy.Raster(Cop_DEM)

    CTSEDEM=os.path.join(CTSEFolder,f"{areaLabel}_Proposed_AllAreas.tif")
    CTSEDEM=arcpy.Raster(CTSEDEM)

    FABDEM=os.path.join(os.path.join(sharedFolder,"FABDEM"),f"FAB_WGS84_{areaLabel}.tif")
    FABDEM=arcpy.Raster(FABDEM)

    FathomDEM=os.path.join(os.path.join(sharedFolder,"FathomDEM"),f"Fathom_{areaLabel}.tif")
    FathomDEM=arcpy.Raster(FathomDEM)*0.01

    GEDTM30=os.path.join(os.path.join(sharedFolder,"GEDTM30"),f"GEDTM30_{areaLabel}.tif")
    GEDTM30=arcpy.Raster(GEDTM30)

    FusionDEM=os.path.join(FusedGDEM_Folder,f"Classification_GDTMTopOther_{areaLabel}_AllAreas.tif")

    
    FusionDEM=arcpy.Raster(FusionDEM)

    run(Cop_DEM, FusionDEM, CTSEDEM,FABDEM,FathomDEM,GEDTM30,ATL08Points)

    print()


IA 1 is working....
 --RMSE-- Raw: 17.90  FusionDEM:10.33  CTSEDEM: 10.41    FathomDEM: 10.76   FAB: 11.17    GEDTM30: 11.62 

IA 2 is working....
 --RMSE-- Raw: 15.98  FusionDEM:10.80  CTSEDEM: 10.77    FathomDEM: 11.64   FAB: 12.05    GEDTM30: 12.19 

IA 3 is working....
 --RMSE-- Raw: 16.56  FusionDEM:10.81  CTSEDEM: 10.36    FathomDEM: 11.20   FAB: 12.10    GEDTM30: 12.18 

IA 4 is working....
 --RMSE-- Raw: 23.46  FusionDEM:11.35  CTSEDEM: 11.16    FathomDEM: 13.32   FAB: 12.37    GEDTM30: 13.03 

IA 5 is working....
 --RMSE-- Raw: 26.70  FusionDEM:15.95  CTSEDEM: 15.62    FathomDEM: 17.04   FAB: 17.47    GEDTM30: 18.20 

IA 6 is working....
 --RMSE-- Raw: 27.12  FusionDEM:14.69  CTSEDEM: 14.88    FathomDEM: 16.31   FAB: 15.82    GEDTM30: 17.46 

IA 7 is working....
 --RMSE-- Raw: 20.88  FusionDEM:11.56  CTSEDEM: 11.54    FathomDEM: 12.78   FAB: 13.87    GEDTM30: 12.71 

IA 8 is working....
 --RMSE-- Raw: 22.07  FusionDEM:14.54  CTSEDEM: 14.10    FathomDEM: 14.79   FAB: 15.20    G

In [ ]:
for k, areaLabel in  enumerate(Area_list) :

    print(f"IA {k+1} is working....")

    ATL08Points=os.path.join(ATL08Folder,f"GEDI_{areaLabel}_Distance30.shp")

    Cop_DEM=os.path.join(os.path.join(sharedFolder,"CopDEM"),f"Cop_WGS84_{areaLabel}.tif")
    Cop_DEM=arcpy.Raster(Cop_DEM)

    CTSEDEM=os.path.join(CTSEFolder,f"{areaLabel}_Proposed_AllAreas.tif")
    CTSEDEM=arcpy.Raster(CTSEDEM)

    FABDEM=os.path.join(os.path.join(sharedFolder,"FABDEM"),f"FAB_WGS84_{areaLabel}.tif")
    FABDEM=arcpy.Raster(FABDEM)

    FathomDEM=os.path.join(os.path.join(sharedFolder,"FathomDEM"),f"Fathom_{areaLabel}.tif")
    FathomDEM=arcpy.Raster(FathomDEM)*0.01

    GEDTM30=os.path.join(os.path.join(sharedFolder,"GEDTM30"),f"GEDTM30_{areaLabel}.tif")
    GEDTM30=arcpy.Raster(GEDTM30)

    FusionDEM=os.path.join(FusedGDEM_Folder,f"RegressionCoeff_TopOther_{areaLabel}_AllAreas.tif")
    
    FusionDEM=arcpy.Raster(FusionDEM)

    run(Cop_DEM, FusionDEM, CTSEDEM,FABDEM,FathomDEM,GEDTM30,ATL08Points)

    print()


IA 1 is working....
 --RMSE-- Raw: 17.90  FusionDEM:10.18  CTSEDEM: 10.41    FathomDEM: 10.76   FAB: 11.17    GEDTM30: 11.62 

IA 2 is working....
 --RMSE-- Raw: 15.98  FusionDEM:10.62  CTSEDEM: 10.77    FathomDEM: 11.64   FAB: 12.05    GEDTM30: 12.19 

IA 3 is working....
 --RMSE-- Raw: 16.56  FusionDEM:10.39  CTSEDEM: 10.36    FathomDEM: 11.20   FAB: 12.10    GEDTM30: 12.18 

IA 4 is working....
 --RMSE-- Raw: 23.46  FusionDEM:11.07  CTSEDEM: 11.16    FathomDEM: 13.32   FAB: 12.37    GEDTM30: 13.03 

IA 5 is working....
 --RMSE-- Raw: 26.70  FusionDEM:15.53  CTSEDEM: 15.62    FathomDEM: 17.04   FAB: 17.47    GEDTM30: 18.20 

IA 6 is working....
 --RMSE-- Raw: 27.12  FusionDEM:14.62  CTSEDEM: 14.88    FathomDEM: 16.31   FAB: 15.82    GEDTM30: 17.46 

IA 7 is working....
 --RMSE-- Raw: 20.88  FusionDEM:11.49  CTSEDEM: 11.54    FathomDEM: 12.78   FAB: 13.87    GEDTM30: 12.71 

IA 8 is working....
 --RMSE-- Raw: 22.07  FusionDEM:14.02  CTSEDEM: 14.10    FathomDEM: 14.79   FAB: 15.20    G